# raw

In [0]:
raw_file=spark.read.format("csv").option("header","true").option("inferSchema","true").load('abfss://devcontainer@discoveryprodfix.dfs.core.windows.net/raw/customer')

In [0]:
raw_file.display()

In [0]:

raw_file.write.format("delta").mode("overwrite").saveAsTable("shivasai_catalog.bronze.cust_stage")

In [0]:
%sql
select * from shivasai_catalog.bronze.cust_stage

In [0]:
%sql
create table if not exists shivasai_catalog.silver.cust_target
(
    cust_id string,
    cust_name string,
    address string,
    start_dt date, 
    end_dt date,
    isActive string
)
using delta; 

In [0]:
%sql

merge into shivasai_catalog.silver.cust_target tgt 

using
(
select
cust_id as mergekey, 
*
from shivasai_catalog.bronze.cust_stage cs 

union all 

select 
NULL as mergekey, 
cs.*
from  shivasai_catalog.silver.cust_target c
inner join shivasai_catalog.bronze.cust_stage cs 
on c.cust_id = cs.cust_id 
and c.address <> cs.address 
and c.isActive = 'Y'

)src 
on src.mergekey = tgt.cust_id 
when matched and src.address <> tgt.address then 
update set tgt.end_dt = current_date()-1 , tgt.isActive='N'
when not matched then 
insert 
(
tgt.cust_id,
tgt.cust_name,
tgt.address,
tgt.start_dt, 
tgt.end_dt,
tgt.isActive
)
values
(
src.cust_id,
src.cust_name,
src.address,
current_date(), 
NULL,
'Y'
)


In [0]:
%sql
select * from shivasai_catalog.silver.cust_target 

In [0]:
dbutils.fs.mkdirs("abfss://devcontainer@discoveryprodfix.dfs.core.windows.net/archiev")


In [0]:
files_list = dbutils.fs.ls("abfss://devcontainer@discoveryprodfix.dfs.core.windows.net/raw/customer")

for files in files_list:
    print(files.path)
    dbutils.fs.mv(files.path, "abfss://devcontainer@discoveryprodfix.dfs.core.windows.net/archiev")

In [0]:
dbutils.fs.ls("abfss://devcontainer@discoveryprodfix.dfs.core.windows.net/archiev")